In [ ]:
# Public-release setup: run from any working directory.
from pathlib import Path
import sys

def find_release_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").is_dir() and (candidate / "docs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the public release directory.")

PROJECT_ROOT = find_release_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
RESULTS_ROOT = PROJECT_ROOT / "results"  # User-supplied artifacts; not included in the release.
FIGURES_ROOT = PROJECT_ROOT / "figures"


# Paper Scatter Figures: Entropy vs. Token Probability / AOFP-L2

This notebook generates the paper-ready single-panel scatter plots requested from the original logits data used by `find_correct_metrics-all_model_raw.ipynb`.

For each available model it writes two figures:

1. `Entropy` vs. `pi(y|s)`
2. `Entropy` vs. `AOFP-L2`

Entropy is always on the y-axis. The notebook first checks which model files exist and only plots models with all required SFT/RL logits files.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from utils.utils_analyze import top12_prob_diff

warnings.filterwarnings("ignore")

# Keep plots paper-friendly and deterministic.
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
# Set DATA_ROOT below to a directory containing the supplied logit artifacts.
PROJECT_ROOT = find_release_root()
DATA_ROOT = PROJECT_ROOT / "results" / "basic_logits_analysis01"
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Expected logits artifacts under {DATA_ROOT}.")
FIGURE_DIR = PROJECT_ROOT / "figures" / "paper_entropy_metrics"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SCATTER_CSV_DIR = PROJECT_ROOT / "results" / "paper_entropy_metric_scatters_csv"
SCATTER_CSV_DIR.mkdir(parents=True, exist_ok=True)
CSV_COMPRESSION = "gzip"  # Writes .csv.gz files; pd.read_csv reads them directly.

# Models explicitly used in find_correct_metrics-all_model_raw.ipynb.
NOTEBOOK_MODELS = [
    "Qwen2.5-0.5B",
    "Qwen3.5-0.8B",
    "Qwen2.5-1.5B",
    "Qwen2.5-7B-Instruct",
    "Llama-3.2-3B",
    "Mistral-7B-v0.1",
]
# Paper model set only.
MODELS = NOTEBOOK_MODELS

EXPERIMENTS = {
    "sft": {
        "path": ("sft", "sft_logits.pt"),
        "logits_key": "sft_logits",
        "labels_key": "sft_labels",
        "label": "SFT Supervision",
        "color": "#1f77b4",
    },
    "rl_tmp1.0": {
        "path": ("rl", "rl_tmp1.0_logits.pt"),
        "logits_key": "rl_logits",
        "labels_key": "rl_labels",
        "label": r"RL $\tau=1.0$",
        "color": "#ff7f0e",
    },
    "rl_tmp0.7": {
        "path": ("rl", "rl_tmp0.7_logits.pt"),
        "logits_key": "rl_logits",
        "labels_key": "rl_labels",
        "label": r"RL $\tau=0.7$",
        "color": "#2ca02c",
    },
    "rl_greedy": {
        "path": ("rl", "rl_greedy_logits.pt"),
        "logits_key": "rl_logits",
        "labels_key": "rl_labels",
        "label": "RL greedy",
        "color": "#d62728",
    },
}

print(f"DATA_ROOT = {DATA_ROOT.resolve() if DATA_ROOT.exists() else DATA_ROOT}")
print(f"FIGURE_DIR = {FIGURE_DIR.resolve()}")
print(f"SCATTER_CSV_DIR = {SCATTER_CSV_DIR.resolve()}")

In [ ]:
DATA_ROOT

In [ ]:
def expected_file(model: str, exp_name: str) -> Path:
    subdir, filename = EXPERIMENTS[exp_name]["path"]
    return DATA_ROOT / model / subdir / filename

availability_rows = []
for model in MODELS:
    row = {"model": model}
    for exp_name in EXPERIMENTS:
        path = expected_file(model, exp_name)
        row[f"{exp_name}_exists"] = path.exists()
        row[f"{exp_name}_path"] = str(path)
    row["all_required_exists"] = all(row[f"{exp_name}_exists"] for exp_name in EXPERIMENTS)
    availability_rows.append(row)

availability = pd.DataFrame(availability_rows)
availability[["model", "all_required_exists"] + [f"{e}_exists" for e in EXPERIMENTS]]

In [ ]:
AVAILABLE_MODELS = availability.loc[availability["all_required_exists"], "model"].tolist()
MISSING_MODELS = availability.loc[~availability["all_required_exists"], "model"].tolist()

# After the compact CSVs have been exported, the original .pt files may be deleted.
# Use CSV_AVAILABLE_MODELS / PLOT_MODELS for plotting, and AVAILABLE_MODELS only for .pt conversion.
def existing_scatter_csv_models(models=MODELS):
    csv_models = []
    for model in models:
        gzip_path = SCATTER_CSV_DIR / f"{model}_entropy_metric_scatter.csv.gz"
        plain_path = SCATTER_CSV_DIR / f"{model}_entropy_metric_scatter.csv"
        if gzip_path.exists() or plain_path.exists():
            csv_models.append(model)
    return csv_models

CSV_AVAILABLE_MODELS = existing_scatter_csv_models()
PLOT_MODELS = CSV_AVAILABLE_MODELS if CSV_AVAILABLE_MODELS else AVAILABLE_MODELS

print(f"Available complete .pt models: {len(AVAILABLE_MODELS)}")
for m in AVAILABLE_MODELS:
    print(f"  pt ok: {m}")

print(f"\nAvailable compact CSV models: {len(CSV_AVAILABLE_MODELS)}")
for m in CSV_AVAILABLE_MODELS:
    print(f"  csv ok: {m}")

if MISSING_MODELS and not CSV_AVAILABLE_MODELS:
    print("\nModels skipped because at least one required logits file is missing:")
    for _, row in availability.loc[~availability["all_required_exists"]].iterrows():
        missing = [e for e in EXPERIMENTS if not row[f"{e}_exists"]]
        print(f"  missing {missing}: {row['model']}")

print(f"\nPLOT_MODELS = {PLOT_MODELS}")

In [ ]:
def compute_aofp_l2_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """AOFP-L2 = ||pi(.|s) - one_hot(y)||_2^2 = 1 - 2*pi(y|s) + sum_v pi(v|s)^2."""
    probs = F.softmax(logits, dim=-1)
    p_y = probs.gather(dim=-1, index=labels.unsqueeze(-1)).squeeze(-1)
    pi_l2_sq = (probs ** 2).sum(dim=-1)
    return 1.0 - 2.0 * p_y + pi_l2_sq


def compute_entropy_yprob_aofp(logit_sequences, label_sequences) -> dict[str, torch.Tensor]:
    """Compute token-level metrics using the same causal shift convention as the source notebook."""
    entropy_list = []
    yprob_list = []
    aofp_l2_list = []

    for logits, labels in tqdm(list(zip(logit_sequences, label_sequences)), leave=False):
        # Source notebook convention: logits[t] predicts rolled label[t], then final position is dropped.
        shifted_labels = torch.roll(labels, -1, dims=-1)[:-1]
        shifted_logits = logits[:-1]

        probs = F.softmax(shifted_logits, dim=-1)
        entropy = -torch.sum(probs * torch.log(probs + 1e-10), dim=-1)
        _, _, _, yprob = top12_prob_diff(shifted_logits, shifted_labels)
        aofp_l2 = compute_aofp_l2_from_logits(shifted_logits, shifted_labels)

        entropy_list.append(entropy.detach().float().cpu())
        yprob_list.append(yprob.detach().float().cpu())
        aofp_l2_list.append(aofp_l2.detach().float().cpu())

    return {
        "entropy": torch.cat(entropy_list, dim=0),
        "yprob": torch.cat(yprob_list, dim=0),
        "aofp_l2": torch.cat(aofp_l2_list, dim=0),
    }


def load_experiment_metrics(model: str, exp_name: str) -> dict[str, torch.Tensor]:
    spec = EXPERIMENTS[exp_name]
    path = expected_file(model, exp_name)
    raw = torch.load(path, map_location="cpu")
    return compute_entropy_yprob_aofp(raw[spec["logits_key"]], raw[spec["labels_key"]])


def subsample_for_plot(x, y, max_points=120_000, seed=0):
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if max_points is not None and len(x) > max_points:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(x), size=max_points, replace=False)
        x = x[idx]
        y = y[idx]
    return x, y


def scatter_csv_path(model: str, output_dir: Path = SCATTER_CSV_DIR) -> Path:
    suffix = ".csv.gz" if CSV_COMPRESSION == "gzip" else ".csv"
    return output_dir / f"{model}_entropy_metric_scatter{suffix}"


def metrics_by_exp_to_dataframe(model: str, metrics_by_exp: dict[str, dict[str, torch.Tensor]]) -> pd.DataFrame:
    """Convert already-computed tensors into a compact scatter-ready table."""
    frames = []
    for exp_name, metrics in metrics_by_exp.items():
        n = int(metrics["entropy"].numel())
        frame = pd.DataFrame({
            "model": model,
            "experiment": exp_name,
            "experiment_label": EXPERIMENTS[exp_name]["label"],
            "token_index": np.arange(n, dtype=np.int64),
            "entropy": metrics["entropy"].numpy().astype(np.float32, copy=False),
            "yprob": metrics["yprob"].numpy().astype(np.float32, copy=False),
            "aofp_l2": metrics["aofp_l2"].numpy().astype(np.float32, copy=False),
        })
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def export_model_scatter_csv(model: str, metrics_by_exp: dict[str, dict[str, torch.Tensor]], output_dir: Path = SCATTER_CSV_DIR) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    df = metrics_by_exp_to_dataframe(model, metrics_by_exp)
    path = scatter_csv_path(model, output_dir)
    df.to_csv(path, index=False, compression=CSV_COMPRESSION)
    print(f"saved {path} ({len(df):,} rows)")
    return path


def build_or_load_model_scatter_csv(model: str, force: bool = False) -> Path:
    """Read large .pt logits only if the compact scatter CSV does not already exist."""
    path = scatter_csv_path(model)
    if path.exists() and not force:
        print(f"exists {path}; skipping .pt load")
        return path

    print(f"\nLoading .pt logits for {model}")
    metrics_by_exp = {}
    for exp_name in EXPERIMENTS:
        pt_path = expected_file(model, exp_name)
        print(f"  {exp_name}: {pt_path}")
        metrics_by_exp[exp_name] = load_experiment_metrics(model, exp_name)
    return export_model_scatter_csv(model, metrics_by_exp)


def load_scatter_csv(model: str) -> pd.DataFrame:
    path = scatter_csv_path(model)
    if not path.exists():
        raise FileNotFoundError(f"Missing compact scatter CSV: {path}. Run build_or_load_model_scatter_csv first.")
    return pd.read_csv(path)


def load_all_scatter_csvs(models=PLOT_MODELS) -> pd.DataFrame:
    frames = [load_scatter_csv(model) for model in models]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()



In [ ]:
PLOT_CONFIG = {
    "figsize": (4.2, 3.4),
    "max_points": 120_000,
    "scatter_size": 20,
    "scatter_alpha": 0.4,
    "legend_markerscale": 1.5,
    "save_pad_inches": 0.03,
}


def style_axis(ax, xlabel):
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Entropy")
    ax.grid(True, linestyle="--", linewidth=0.45, alpha=0.35)


def draw_extra_lines(ax, line_extra=None):
    if line_extra == 1:
        ax.plot([0.0, 0.065], [0.95, 0.0], "k--", lw=1.8, alpha=0.8, zorder=10)
        ax.plot([0.0, 0.39], [4.25, 0.0], "k--", lw=1.8, alpha=0.8, zorder=10)
    if line_extra == 2:
        ax.axvline(x=0.8, color="black", linestyle="--", linewidth=1.8, alpha=0.8, zorder=10)
        ax.axvline(x=1.8, color="black", linestyle="--", linewidth=1.8, alpha=0.8, zorder=10)
        ax.text(0.4, 0.96, "Confident Tokens", transform=ax.get_xaxis_transform(),
                ha="center", va="top", fontsize=8)
        
        ax.text(1.30, 0.8, "Semantic Forkings", transform=ax.get_xaxis_transform(),
                ha="center", va="top", fontsize=8)
        
        ax.text(2.0, 0.4, "Extreme\n Energy", transform=ax.get_xaxis_transform(),
                ha="center", va="top", fontsize=8)


def save_single_scatter(
    model,
    metrics_by_exp,
    x_key,
    xlabel,
    filename_suffix,
    max_points=None,
    plot_config=PLOT_CONFIG,
    line_extra=None,
):
    """Plot one already-loaded tensor cache. Kept for compatibility with older cells."""
    if max_points is None:
        max_points = plot_config["max_points"]

    fig, ax = plt.subplots(figsize=plot_config["figsize"])
    for exp_i, (exp_name, spec) in enumerate(EXPERIMENTS.items()):
        metrics = metrics_by_exp[exp_name]
        x, y = subsample_for_plot(metrics[x_key].numpy(), metrics["entropy"].numpy(), max_points=max_points, seed=1234 + exp_i)
        ax.scatter(x, y, s=plot_config["scatter_size"], alpha=plot_config["scatter_alpha"], linewidths=0, color=spec["color"], label=spec["label"], rasterized=True)
    style_axis(ax, xlabel)
    draw_extra_lines(ax, line_extra=line_extra)
    if line_extra != 2:
        ax.legend(frameon=False, markerscale=plot_config["legend_markerscale"], handletextpad=0.25)
    fig.tight_layout()

    safe_model = model.replace("/", "_")
    png_path = FIGURE_DIR / f"{safe_model}_{filename_suffix}.png"
    pdf_path = FIGURE_DIR / f"{safe_model}_{filename_suffix}.pdf"
    fig.savefig(png_path, bbox_inches="tight", pad_inches=plot_config["save_pad_inches"])
    fig.savefig(pdf_path, bbox_inches="tight", pad_inches=plot_config["save_pad_inches"])
    plt.show()
    return png_path, pdf_path


def save_single_scatter_from_df(
    model,
    scatter_df,
    x_key,
    xlabel,
    filename_suffix,
    max_points=None,
    plot_config=PLOT_CONFIG,
    line_extra=None,
):
    """Plot one model from compact CSV/DataFrame data. No .pt files are read here."""
    if max_points is None:
        max_points = plot_config["max_points"]

    fig, ax = plt.subplots(figsize=plot_config["figsize"])
    model_df = scatter_df.loc[scatter_df["model"] == model]
    for exp_i, (exp_name, spec) in enumerate(EXPERIMENTS.items()):
        sub = model_df.loc[model_df["experiment"] == exp_name]
        if sub.empty:
            warnings.warn(f"No compact scatter rows for {model} / {exp_name}")
            continue
        x, y = subsample_for_plot(sub[x_key].to_numpy(), sub["entropy"].to_numpy(), max_points=max_points, seed=1234 + exp_i)
        ax.scatter(x, y, s=plot_config["scatter_size"], alpha=plot_config["scatter_alpha"], linewidths=0, color=spec["color"], label=spec["label"], rasterized=True)
    style_axis(ax, xlabel)
    draw_extra_lines(ax, line_extra=line_extra)
    if line_extra != 2:
        ax.legend(frameon=False, markerscale=plot_config["legend_markerscale"], handletextpad=0.25)
    fig.tight_layout()

    safe_model = model.replace("/", "_")
    png_path = FIGURE_DIR / f"{safe_model}_{filename_suffix}.png"
    pdf_path = FIGURE_DIR / f"{safe_model}_{filename_suffix}.pdf"
    fig.savefig(png_path, bbox_inches="tight", pad_inches=plot_config["save_pad_inches"])
    fig.savefig(pdf_path, bbox_inches="tight", pad_inches=plot_config["save_pad_inches"])
    plt.show()
    return png_path, pdf_path


def plot_all_paper_scatters(metrics_cache, models=AVAILABLE_MODELS, plot_config=PLOT_CONFIG, line_extra=None):
    """Generate all paper figures from the preloaded tensor metrics cache."""
    outputs = []
    for model in models:
        metrics_by_exp = metrics_cache[model]
        outputs.extend(save_single_scatter(model, metrics_by_exp, x_key="yprob", xlabel=r"$\pi(y\mid s)$", filename_suffix="entropy_vs_yprob", plot_config=plot_config, line_extra=line_extra))
        outputs.extend(save_single_scatter(model, metrics_by_exp, x_key="aofp_l2", xlabel=r"$\|g\|_2^2$", filename_suffix="entropy_vs_aofp_l2", plot_config=plot_config, line_extra=line_extra))
    return [str(path) for path in outputs]


def plot_all_paper_scatters_from_df(scatter_df, models=AVAILABLE_MODELS, plot_config=PLOT_CONFIG, line_extra=None):
    """Generate all paper figures from compact CSV/DataFrame data."""
    outputs = []
    for model in models:
        outputs.extend(save_single_scatter_from_df(model, scatter_df, x_key="yprob", xlabel=r"$\pi(y\mid s)$", filename_suffix="entropy_vs_yprob", plot_config=plot_config, line_extra=line_extra))
        outputs.extend(save_single_scatter_from_df(model, scatter_df, x_key="aofp_l2", xlabel=r"$\|g\|_2^2$", filename_suffix="entropy_vs_aofp_l2", plot_config=plot_config, line_extra=line_extra))
    return [str(path) for path in outputs]

In [ ]:
# One-time conversion from large .pt logits to compact scatter CSVs.
# Set FORCE_REBUILD_CSV = True only if the .pt files changed and you want to overwrite CSVs.
# If the .pt files have already been deleted, skip this cell and use the compact CSV loading cell below.

FORCE_REBUILD_CSV = False
scatter_csv_paths = []
for model in AVAILABLE_MODELS:
    scatter_csv_paths.append(build_or_load_model_scatter_csv(model, force=FORCE_REBUILD_CSV))

scatter_csv_paths

In [ ]:
# Load compact CSVs for plotting. After this cell, plot edits do not touch the large .pt files.

scatter_df = load_all_scatter_csvs(PLOT_MODELS)
print(f"loaded compact scatter table: {len(scatter_df):,} rows")
scatter_df.head()

In [ ]:
# Example: plot one model from compact CSV data.
MODEL_ = "Qwen2.5-7B-Instruct"

save_single_scatter_from_df(
    model=MODEL_,
    scatter_df=scatter_df,
    x_key="yprob",
    xlabel=r"$\pi(y\mid s)$",
    filename_suffix="entropy_vs_yprob",
    max_points=None,
    plot_config=PLOT_CONFIG,
    line_extra=1,
)

In [ ]:
save_single_scatter_from_df(
    model=MODEL_,
    scatter_df=scatter_df,
    x_key="aofp_l2",
    xlabel=r"$\|g\|_2^2$",
    filename_suffix="entropy_vs_aofp_l2",
    max_points=None,
    plot_config=PLOT_CONFIG,
    line_extra=2,
)

In [ ]:
# Generate all requested paper figures from compact CSV data.
# This cell does not load .pt files. Edit PLOT_CONFIG / style_axis / save_single_scatter_from_df above,
# then rerun this cell to update figure styling.

# all_outputs = plot_all_paper_scatters_from_df(scatter_df, line_extra=None)

# print("\nSaved figures:")
# for path in all_outputs:
#     print(path)

In [ ]:
# Optional: write a small manifest for the compact CSVs and paper figure outputs.
manifest = {
    "data_root": str(DATA_ROOT),
    "figure_dir": str(FIGURE_DIR),
    "scatter_csv_dir": str(SCATTER_CSV_DIR),
    "available_models": AVAILABLE_MODELS,
    "missing_models": MISSING_MODELS,
    "scatter_csvs": [str(p) for p in scatter_csv_paths] if "scatter_csv_paths" in globals() else [],
    "figures": all_outputs if "all_outputs" in globals() else [],
}
manifest_path = FIGURE_DIR / "paper_entropy_metric_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest_path

## Appendix: All-Model Scatter Grid

This section draws one appendix figure with six model columns and two rows. The top row is `Entropy` vs. `pi(y|s)`, and the bottom row is `Entropy` vs. `||g||_2^2` / AOFP-L2. It uses `scatter_df`, so run the compact-CSV loading cell first.

In [ ]:
APPENDIX_GRID_CONFIG = {
    "figsize": (18.0, 6.0),
    "max_points_per_experiment": 10_000,
    "scatter_size": 15,
    "scatter_alpha": 0.25,
    "save_pad_inches": 0.04,
}


def plot_appendix_all_model_grid(
    scatter_df,
    models=PLOT_MODELS,
    plot_config=APPENDIX_GRID_CONFIG,
    filename_stem="appendix_entropy_metric_all_models_grid",
):
    fig, axes = plt.subplots(2, len(models), figsize=plot_config["figsize"], sharey="row")
    x_specs = [
        ("yprob", r"$\pi(y\mid s)$"),
        ("aofp_l2", r"$\|g\|_2^2$"),
    ]

    for col, model in enumerate(models):
        model_df = scatter_df.loc[scatter_df["model"] == model]
        for row, (x_key, xlabel) in enumerate(x_specs):
            ax = axes[row, col]
            for exp_i, (exp_name, spec) in enumerate(EXPERIMENTS.items()):
                sub = model_df.loc[model_df["experiment"] == exp_name]
                if sub.empty:
                    warnings.warn(f"No compact scatter rows for {model} / {exp_name}")
                    continue
                x, y = subsample_for_plot(
                    sub[x_key].to_numpy(),
                    sub["entropy"].to_numpy(),
                    max_points=plot_config["max_points_per_experiment"],
                    seed=4321 + 17 * col + exp_i,
                )
                ax.scatter(
                    x,
                    y,
                    s=plot_config["scatter_size"],
                    alpha=plot_config["scatter_alpha"],
                    linewidths=0,
                    color=spec["color"],
                    label=spec["label"],
                    rasterized=True,
                )
            ax.set_xlabel(xlabel)
            if col == 0:
                ax.set_ylabel("Entropy")
            else:
                ax.set_ylabel("")
            if row == 0:
                ax.set_title(model)
            ax.grid(True, linestyle="--", linewidth=0.35, alpha=0.3)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, markerscale=3, loc="upper center", bbox_to_anchor=(0.5, 1.02), ncol=len(EXPERIMENTS), frameon=False)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    png_path = FIGURE_DIR / f"{filename_stem}.png"
    pdf_path = FIGURE_DIR / f"{filename_stem}.pdf"
    fig.savefig(png_path,dpi=150, bbox_inches="tight", pad_inches=plot_config["save_pad_inches"])
    fig.savefig(pdf_path,dpi=150, bbox_inches="tight", pad_inches=plot_config["save_pad_inches"])
    plt.show()
    print(f"saved {png_path}")
    print(f"saved {pdf_path}")
    return png_path, pdf_path


appendix_outputs = plot_appendix_all_model_grid(scatter_df)
appendix_outputs